# London May–September Bedroom LME Diagnostics

Diagnostic plots for models M1-M15 fitted by `R/efus_mm_london_bedroom_4month.R`.

Three model families:
- **Static (M1-M7)**: LME with no residual-correlation structure
- **AR(1) (M8-M9)**: LME with AR(1) residual correlation
- **SAR(1,24) (M11-M15)**: LME with seasonal AR(1,24) residual correlation

Time-of-day treatments include none, sin/cos harmonics, and hour-of-day dummies; selected models also add binary and categorical building characteristics.

**Normalised residuals**: ACF, QQ, and residual-vs-fitted plots use `nlme` normalised residuals, i.e. residuals transformed by the fitted within-group residual-correlation structure and scaled by the model residual standard deviation. For static models these are essentially standardised conditional response residuals. For AR(1) and SAR(1,24) models they are approximately whitened residuals; remaining autocorrelation in these residuals indicates that the fitted residual-correlation structure has not fully captured the serial dependence.

**OSA**: One-step-ahead predictions incorporate the lagged AR state:
OSA_t = mean_t + rho*r_{t-1} [+ Phi*r_{t-24}], where r_{t-k} are response residuals.


In [ ]:
# Formatting of plots

import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
import os 

plt.style.use(['science', 'nature','bright'])
mpl.rcParams['savefig.format'] = 'svg'
os.makedirs('plots/efus2017/london_bedroom_summer', exist_ok=True)


In [ ]:
import os
# Paths below are relative to the project root (this notebook's folder);
# Jupyter starts the kernel there, so no chdir is needed.
assert os.path.isdir('diagnostics'), \
    'diagnostics/ not found: start Jupyter from the project root and run the R scripts first'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.patches import Patch

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})
plt.rcParams['text.usetex'] = False

# ── Model metadata ─────────────────────────────────────────────────────────────
MODELS = ["M1", "M2", "M3", "M4", "M5", "M6", "M7",
          "M8", "M9", "M11", "M12", "M13", "M14", "M15"]
FAMILY = {
    "M1": "Static",  "M2": "Static",  "M3": "Static",
    "M4": "Static",  "M5": "Static",  "M6": "Static",  "M7": "Static",
    "M8": "AR(1)",   "M9": "AR(1)",
    "M11": "SAR(1,24)", "M12": "SAR(1,24)", "M13": "SAR(1,24)",
    "M14": "SAR(1,24)", "M15": "SAR(1,24)",
}
TODS = {
    "M1": "none",          "M2": "sin/cos",        "M3": "hour",
    "M4": "hour+bin",      "M5": "hour+bin+cat",
    "M6": "sin/cos+bin",   "M7": "sin/cos+bin+cat",
    "M8": "none",          "M9": "sin/cos",
    "M11": "none",         "M12": "sin/cos",       "M13": "hour",
    "M14": "hour+bin+cat", "M15": "sin/cos+bin+cat",
}
FAMILY_COLOUR = {"Static": "#2E67D0", "AR(1)": "#27C05A", "SAR(1,24)": "#FD6500"}
NROWS, NCOLS = 4, 4

# ── Hardcoded fallbacks (values from R/efus_mm_london_bedroom_4month.R run) ──────────────
AIC_VALUES = {
    "M1": 178513.59, "M2": 166118.70, "M3": 165802.86,
    "M4": 165806.52, "M5": 165806.60, "M6": 166122.35, "M7": 166122.41,
    "M8":  10868.82, "M9":   4270.16,
    "M11": 89962.80, "M12": 85930.50, "M13": 85463.18,
    "M14": float("nan"), "M15": float("nan"),
}
AR_PARAMS = {
    "M1":  (None, None), "M2":  (None, None), "M3":  (None, None),
    "M4":  (None, None), "M5":  (None, None), "M6":  (None, None), "M7": (None, None),
    "M8":  (0.9915, None), "M9":  (0.9927, None),
    "M11": (0.7700, 0.1136), "M12": (0.7740, 0.1088), "M13": (0.7752, 0.1078),
    "M14": (0.7752, 0.1078), "M15": (0.7740, 0.1088),
}
SIG_EPS = {
    "M1": 1.5690, "M2": 1.3768, "M3": 1.3716,
    "M4": 1.3716, "M5": 1.3716, "M6": 1.3768, "M7": 1.3768,
    "M8":  10868.82, "M9":   4270.16,
    "M11": 0.6196, "M12": 0.5938, "M13": 0.5906,
    "M14": 0.5906, "M15": 0.5938,
}

# Override with CSV exports if available (automatically updated after each R run)
_aic_path = "diagnostics/london_may_sep_bedroom_aic.csv"
_vc_path  = "diagnostics/london_may_sep_bedroom_varcomp.csv"
if os.path.exists(_aic_path):
    _aic = pd.read_csv(_aic_path)
    AIC_VALUES.update(dict(zip(_aic["model"], _aic["aic"])))
if os.path.exists(_vc_path):
    _vc = pd.read_csv(_vc_path).set_index("model")
    for m in MODELS:
        if m in _vc.index:
            rho = None if pd.isna(_vc.loc[m, "rho"]) else float(_vc.loc[m, "rho"])
            phi = None if pd.isna(_vc.loc[m, "phi"]) else float(_vc.loc[m, "phi"])
            AR_PARAMS[m] = (rho, phi)
            SIG_EPS[m]   = float(_vc.loc[m, "sig_eps"])

# ── % variance accounted for vs family-specific null model ──────────────────
# Uses total_obs_var from R: accounts for random-slope contrib and cov(u0,u1)
NULL_FOR = {"Static": "M0_static", "AR(1)": "M0_ar1", "SAR(1,24)": "M0_sar24"}
PVAR = {}
if os.path.exists(_vc_path):
    for m in MODELS:
        if m in _vc.index:
            null_key = NULL_FOR[FAMILY[m]]
            if null_key in _vc.index:
                null_var  = float(_vc.loc[null_key, "total_obs_var"])
                model_var = float(_vc.loc[m,        "total_obs_var"])
                PVAR[m] = 100.0 * (1.0 - model_var / null_var)


# ── Load data ──────────────────────────────────────────────────────────────────
df    = pd.read_parquet("diagnostics/london_may_sep_bedroom_diagnostics.parquet")
df["ts"] = pd.to_datetime(df["hour"])
ranef = pd.read_csv("diagnostics/london_may_sep_bedroom_ranef.csv")
dwellings = df["dwelling"].unique()
print(f"{len(df):,} obs, {len(dwellings)} dwellings")

# ── Shared helpers ─────────────────────────────────────────────────────────────
def _label(ax, nm):
    rho, phi = AR_PARAMS[nm]
    ax.set_title(f"{nm} — {FAMILY[nm]}/{TODS[nm]}", fontsize=7, pad=3)
    if rho is not None:
        lines = [f"$\\rho$={rho:.4f}"]
        if phi is not None:
            lines.append(f"$\\Phi$={phi:.4f}")
        ax.text(0.97, 0.97, "\n".join(lines),
                transform=ax.transAxes, fontsize=6,
                va="top", ha="right"
                ) # bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0.5)

def hide_unused(axes):
    for ax in axes.flat[len(MODELS):]:
        ax.set_visible(False)

MAX_LAG = 48
def mean_acf(col, max_lag=MAX_LAG):
    acfs = []
    for _, grp in df.groupby("dwelling", sort=False):
        grp = grp.sort_values("ts")
        r = grp[col].to_numpy()
        if len(r) < max_lag + 5:
            continue
        r = r - np.nanmean(r)
        if np.any(~np.isfinite(r)):
            continue
        v = np.var(r, ddof=0)
        if v < 1e-12:
            continue
        ac = np.array([
            np.mean(r[:len(r) - k] * r[k:]) / v
            for k in range(max_lag + 1)
        ])
        acfs.append(ac)
    if len(acfs) == 0:
        return np.full(max_lag + 1, np.nan)
    return np.mean(np.vstack(acfs), axis=0)

rng = np.random.default_rng(0)


## 1. AIC Comparison

Marginal AIC across the three model families. AR(1) models are far better than
static; SAR(1,24) is intermediate (captures seasonal structure but not the
strong lag-1 dependence that AR(1) handles).


In [ ]:
# ── load log-likelihoods (for AIC cross-check, not used in pvar) ─────────────
_ll = {}
if os.path.exists(_aic_path):
    _aicdf = pd.read_csv(_aic_path).set_index("model")
    for _m in _aicdf.index:
        _ll[_m] = float(_aicdf.loc[_m, "loglik"])

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.0, 2.9),
    constrained_layout=True,
    sharey=False,
)

x       = np.arange(len(MODELS))
colours = [FAMILY_COLOUR[FAMILY[m]] for m in MODELS]

# ── Left: AIC (log scale) ─────────────────────────────────────────────────
ax_aic = axes[0]
ax_aic.bar(x, [AIC_VALUES[m] for m in MODELS],
           color=colours, edgecolor="black", linewidth=0.4, width=0.8)
ax_aic.set_yscale("log")
ax_aic.set_xticks(x)
ax_aic.set_xticklabels(MODELS, rotation=35, ha="right", fontsize=7)
ax_aic.tick_params(axis="y", labelsize=7)
ax_aic.tick_params(axis="x", which="minor", bottom=False)
ax_aic.set_ylabel("AIC (log scale)", fontsize=8)
ax_aic.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
ax_aic.set_title("AIC", fontsize=8)

# ── Right: % variance accounted for using family-specific nulls ─────────
ax_pv = axes[1]
if PVAR:
    pv = [PVAR.get(m, float("nan")) for m in MODELS]
    ax_pv.bar(x, pv, color=colours, edgecolor="black", linewidth=0.4, width=0.8)
    ax_pv.axhline(0, color="black", linewidth=0.6, linestyle="-")
    ax_pv.set_xticks(x)
    ax_pv.set_xticklabels(MODELS, rotation=35, ha="right", fontsize=7)
    ax_pv.tick_params(axis="y", labelsize=7)
    ax_pv.tick_params(axis="x", which="minor", bottom=False)
    ax_pv.set_ylabel("% variance accounted for vs null", fontsize=8)
    ax_pv.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
    ax_pv.set_title("Observation-level variance: % reduction vs null", fontsize=8)

# ── Shared legend ─────────────────────────────────────────────────────────
handles = [Patch(color=c, label=f) for f, c in FAMILY_COLOUR.items()]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.08),
           ncol=len(handles), fontsize=7, frameon=False)
fig.suptitle("London August LME - AIC & variance accounted for (M1-M15)",
             fontsize=9, y=1.14)

fig.savefig(
    "plots/efus2017/london_bedroom_summer/mm_london_bedroom_aic.svg",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


## 2. Residual ACF: normalized residuals

ACF of nlme normalized residuals.

For **static** models: normalized residuals are standardized conditional residuals
and may still show serial autocorrelation.

For **AR(1)/SAR(1,24)** models: normalized residuals are transformed by the fitted
correlation structure and should be approximately white noise if the residual
model is adequate.

Dashed lines are a rough per-dwelling white-noise reference: ±1.96/√(median
dwelling series length). Dotted vertical line = lag 24.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 8), sharex=True, sharey=True)
lags = np.arange(MAX_LAG + 1)

median_n = df.groupby("dwelling").size().median()
ci = 1.96 / np.sqrt(median_n)

for ax, nm in zip(axes.flat, MODELS):
    col = f"nresid_{nm}"
    if col not in df.columns:
        ax.text(
            0.5, 0.5,
            f"{col}\nnot found",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=8,
        )
        _label(ax, nm)
        continue
    acf = mean_acf(col)
    c = FAMILY_COLOUR[FAMILY[nm]]
    ax.bar(lags, acf, color=c, alpha=0.75, width=0.8)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axhline( ci, color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.axhline(-ci, color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.axvline(24, color="black", linewidth=0.5, linestyle=":", alpha=0.5)
    _label(ax, nm)
    ax.set_xlim(-0.5, MAX_LAG + 0.5)
    ax.set_ylim(-0.25, 1.0)
    ax.set_xlabel("Lag (hours)", fontsize=8)

hide_unused(axes)
for ax in axes[:, 0]:
    ax.set_ylabel("Mean ACF", fontsize=8)

fig.suptitle(
    "London August — Mean ACF of nlme normalized residuals\n"
    "Dashed = rough per-dwelling white-noise reference (±1.96/√n). Dotted = lag 24 hrs.",
    fontsize=10,
)
fig.tight_layout()

fig.savefig(
    "plots/efus2017/london_bedroom_summer/mm_london_bedroom_acf.svg",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


## 3. Normal QQ Plots (Normalised Residuals)

Tests the marginal distribution of the `nlme` normalised residuals. Deviations from the diagonal indicate heavy tails, skew, outliers, or remaining model misspecification after applying the fitted residual-correlation structure.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 10))

lims = [-5, 5]

for ax, nm in zip(axes.flat, MODELS):
    col = f"nresid_{nm}"

    r = (
        df[col].dropna().values
        if col in df.columns
        else df[f"resid_{nm}"].dropna().values
    )

    if len(r) > 5000:
        r = rng.choice(r, 5000, replace=False)

    (osm, osr), (slope, intercept, _) = stats.probplot(r, dist="norm")

    c = FAMILY_COLOUR[FAMILY[nm]]

    ax.scatter(
        osm, osr,
        s=3,
        alpha=0.4,
        color=c,
        rasterized=True,
    )

    x_line = np.array(lims)
    ax.plot(
        x_line,
        slope * x_line + intercept,
        color="black",
        linewidth=1,
    )

    _label(ax, nm)

    ax.set_xlabel("Theoretical quantiles", fontsize=7)
    ax.set_ylabel("Sample quantiles", fontsize=7)
    ax.tick_params(labelsize=7)

    # identical scales
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    # square panels
    ax.set_aspect("equal", adjustable="box")

hide_unused(axes)

fig.suptitle(
    "London August - Normal QQ of nlme normalised residuals",
    fontsize=11,
)

fig.tight_layout()

fig.savefig(
    "plots/efus2017/london_bedroom_summer/mm_london_bedroom_qq.svg",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 4. Normalised Residuals vs Fitted Values

Checks for heteroskedasticity or nonlinear mean structure. Red = running mean (window=n/50). Should be flat at zero.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 9))
idx_sample = rng.choice(len(df), min(4000, len(df)), replace=False)

for ax, nm in zip(axes.flat, MODELS):
    fitted = df[f"fitted_{nm}"].values[idx_sample]
    col = f"nresid_{nm}"
    resid = (df[col].values[idx_sample] if col in df.columns
             else df[f"resid_{nm}"].values[idx_sample])
    c = FAMILY_COLOUR[FAMILY[nm]]
    ax.scatter(fitted, resid, s=3, alpha=0.3, color=c, rasterized=True)
    ax.axhline(0, color="black", linewidth=0.8)
    order = np.argsort(fitted)
    fs, rs = fitted[order], resid[order]
    w = max(1, len(fs) // 50)
    smooth = np.convolve(rs, np.ones(w) / w, mode="valid")
    ax.plot(fs[w//2: w//2 + len(smooth)], smooth, color="red", linewidth=1, alpha=0.8)
    _label(ax, nm)
    ax.set_xlabel("Fitted (deg C)", fontsize=7)
    ax.set_ylabel("Norm. resid.", fontsize=7)
    ax.tick_params(labelsize=7)

hide_unused(axes)
fig.suptitle("London August - nlme normalised residuals vs fitted (red = running mean, n=4000)",
             fontsize=11)
fig.tight_layout()
fig.savefig("plots/efus2017/london_bedroom_summer/mm_london_bedroom_resid_fit.png", dpi=150)
plt.show()


## 5. Random Effects (BLUPs)

u₀ = dwelling-level intercept BLUP; u₁ = T_out slope BLUP. Correlation shown in top-left corner.


In [ ]:
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(10, 9))

for ax, nm in zip(axes.flat, MODELS):
    if f"u0_{nm}" not in ranef.columns:
        ax.text(0.5, 0.5, f"u0_{nm}\nnot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=8)
        _label(ax, nm)
        continue
    u0 = ranef[f"u0_{nm}"].values
    u1 = ranef[f"u1_{nm}"].values
    c = FAMILY_COLOUR[FAMILY[nm]]
    ax.scatter(u0, u1, s=25, alpha=0.7, color=c, edgecolors="white", linewidths=0.3)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    if np.std(u0) > 1e-6 and np.std(u1) > 1e-6:
        corr = np.corrcoef(u0, u1)[0, 1]
        ax.text(0.05, 0.95, f"r = {corr:.3f}", transform=ax.transAxes,
                fontsize=7, va="top")
    _label(ax, nm)
    ax.set_xlabel("u0 intercept BLUP (deg C)", fontsize=7)
    ax.set_ylabel("u1 slope BLUP (deg C/deg C)", fontsize=7)
    ax.tick_params(labelsize=7)

hide_unused(axes)
fig.suptitle("London August - Random effects BLUPs (u0 = dwelling intercept, u1 = T_out slope)",
             fontsize=11)
fig.tight_layout()
fig.savefig("plots/efus2017/london_bedroom_summer/mm_london_bedroom_ranef.svg", dpi=150)
plt.show()


## 6. Time Series — One-Step-Ahead Predictions

Shows observed (black), marginal mean (dashed), and one-step-ahead (OSA) prediction (solid)
for one example dwelling (highest variance among median-length dwellings).

The marginal mean is flat relative to the diurnal cycle; the OSA prediction incorporates
the lagged AR state and tracks the slow August warming trend.


In [ ]:
# Choose example dwelling: median obs count, highest T_in variance
obs_counts = df.groupby("dwelling").size()
med = obs_counts.median()
candidates = obs_counts[abs(obs_counts - med) <= 10].index.tolist()
variances = df[df["dwelling"].isin(candidates)].groupby("dwelling")["T_in"].var()
example_dw = variances.idxmax()

dw_df = df[df["dwelling"] == example_dw].sort_values("ts").reset_index(drop=False)
n_show = min(len(dw_df), 21 * 24)
t = np.arange(n_show)

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(12, 8), sharex=True)

for ax, nm in zip(axes.flat, MODELS):
    c = FAMILY_COLOUR[FAMILY[nm]]
    rho, phi = AR_PARAMS[nm]
    fitted_vals = dw_df[f"fitted_{nm}"].values[:n_show]
    resid_vals  = dw_df[f"resid_{nm}"].values

    if rho is not None:
        osa = fitted_vals.copy()
        osa[1:] = fitted_vals[1:] + rho * resid_vals[:n_show - 1]
        if phi is not None:
            osa[24:] = osa[24:] + phi * resid_vals[:n_show - 24]
    else:
        osa = fitted_vals

    obs = dw_df["T_in"].values[:n_show]
    ax.plot(t, obs, color="black", linewidth=0.6, alpha=0.5, label="Observed", zorder=1)
    ax.plot(t, fitted_vals, color=c, linewidth=1.0, linestyle="--",
            alpha=0.6, label="Mean (fitted)", zorder=2)
    if rho is not None:
        ax.plot(t, osa, color=c, linewidth=1.2, label="OSA prediction", zorder=3)
    _label(ax, nm)
    ax.tick_params(labelsize=7)
    ax.set_ylabel("T_in (deg C)", fontsize=7)

hide_unused(axes)
for ax in axes[NROWS - 1]:
    ax.set_xlabel("Hour index (first 21 days)", fontsize=8)

# Collect unique legend handles from all panels
handles, labels = [], []
for ax in axes.flat[:len(MODELS)]:
    for h, l in zip(*ax.get_legend_handles_labels()):
        if l not in labels:
            handles.append(h); labels.append(l)
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8,
           bbox_to_anchor=(0.5, -0.01))

fig.suptitle(
    f"London August - Dwelling {example_dw}: observed (black), marginal mean (dashed),\n"
    "one-step-ahead prediction (solid, AR/SAR models only).",
    fontsize=10)
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("plots/efus2017/london_bedroom_summer/mm_london_bedroom_timeseries.svg",
            dpi=150, bbox_inches="tight")
plt.show()


## 7. Variance Components and Residual-Correlation Parameters

Residual standard deviation scale sigma_epsilon, AR coefficient rho, and seasonal AR coefficient Phi (lag 24). In these `nlme` models, sigma_epsilon is the residual variance scale used with the fitted within-group correlation structure; it should not be labelled as marginal residual standard deviation or as a standalone marginal residual standard deviation in the comparison table.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
x = np.arange(len(MODELS))
colours = [FAMILY_COLOUR[FAMILY[m]] for m in MODELS]

sig_eps = [SIG_EPS[m] for m in MODELS]
axes[0].bar(x, sig_eps, color=colours, edgecolor="white", linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(MODELS, fontsize=8, rotation=45, ha="right")
axes[0].set_title("Residual SD scale sigma_eps (deg C)", fontsize=9)
axes[0].set_ylabel("sigma_eps (deg C)")

rho_vals = [AR_PARAMS[m][0] if AR_PARAMS[m][0] is not None else 0 for m in MODELS]
axes[1].bar(x, rho_vals, color=colours, edgecolor="white", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(MODELS, fontsize=8, rotation=45, ha="right")
axes[1].set_title("AR coefficient rho", fontsize=9)
axes[1].set_ylabel("rho")

phi_vals = [AR_PARAMS[m][1] if AR_PARAMS[m][1] is not None else 0 for m in MODELS]
axes[2].bar(x, phi_vals, color=colours, edgecolor="white", linewidth=0.5)
axes[2].set_xticks(x)
axes[2].set_xticklabels(MODELS, fontsize=8, rotation=45, ha="right")
axes[2].set_title("Seasonal AR coefficient Phi (lag 24)", fontsize=9)
axes[2].set_ylabel("Phi")

handles = [Patch(color=c, label=f) for f, c in FAMILY_COLOUR.items()]
axes[0].legend(handles=handles, fontsize=8)
fig.suptitle("London August - Variance components and correlation parameters (M1-M15)", fontsize=10)
fig.tight_layout()
fig.savefig("plots/efus2017/london_bedroom_summer/mm_london_bedroom_variance_components.png", dpi=150)
plt.show()


## 8. Variance Accounted For by Fixed Model

Percentage of observation-level variance accounted for by the fixed part of each model, calculated as the reduction in total observation-level variance relative to the corresponding constant-only null model from the same residual-correlation family:

$$
\% \text{ variance accounted for}
= 100 \times \left(1 - \frac{V_{\text{model}}}{V_{\text{null}}}\right)
$$

where:

- `M0_static` is the constant-only null for the static models;
- `M0_ar1` is the constant-only null for the AR(1) models;
- `M0_sar24` is the constant-only null for the SAR(1,24) models.

The variance quantity used here is `total_obs_var` exported from R, not the simple sum `sig_u0^2 + sig_u1^2 + sig_eps^2`. For these models,

$$
V_{\text{model}}
= \operatorname{E}\left[\operatorname{Var}(u_0 + u_1 T_{out,c})\right]
+ \sigma^2_\varepsilon,
$$

so the calculation includes the random intercept variance, random slope variance, their covariance, the observed distribution of `T_out_c`, and the residual variance scale from `nlme`.

The fitted null model changes by family so that models are compared only with a null model having the same random-effects and residual-correlation structure. Negative values are retained because they indicate that the fitted model has a larger estimated total observation-level variance than its family-specific null.


In [ ]:
# ── Variance accounted for: observation-level components table ────────────────
_vc_tbl = pd.read_csv("diagnostics/london_may_sep_bedroom_varcomp.csv").set_index("model")

required_cols = {"sig_u0", "sig_u1", "sig_eps", "cov_u01", "total_obs_var"}
missing_cols = required_cols.difference(_vc_tbl.columns)
if missing_cols:
    raise ValueError(
        "diagnostics/london_may_sep_bedroom_varcomp.csv is missing required columns: "
        + ", ".join(sorted(missing_cols))
        + ". Re-run the updated R script."
    )

NULL_FOR = {
    "Static": "M0_static",
    "AR(1)": "M0_ar1",
    "SAR(1,24)": "M0_sar24",
}

NULL_MODELS = [
    ("M0_static", "Static", "-"),
    ("M0_ar1", "AR(1)", "-"),
    ("M0_sar24", "SAR(1,24)", "-"),
]

rows = []

# Null baselines - one per residual-correlation family
for null_key, fam, tod in NULL_MODELS:
    if null_key not in _vc_tbl.index:
        continue

    v = _vc_tbl.loc[null_key]

    rows.append({
        "Model": null_key,
        "Null model": "-",
        "Family": fam,
        "ToD": tod,
        "sigma2_u0": float(v["sig_u0"])**2,
        "sigma2_u1": float(v["sig_u1"])**2,
        "cov_u0u1": float(v["cov_u01"]),
        "sigma2_eps": float(v["sig_eps"])**2,
        "Total obs sigma2": float(v["total_obs_var"]),
        "% accounted for": 0.0,
    })

for m in MODELS:
    if m not in _vc_tbl.index:
        continue

    v = _vc_tbl.loc[m]
    null_key = NULL_FOR[FAMILY[m]]

    if null_key not in _vc_tbl.index:
        raise ValueError(
            f"Null model {null_key} for {m} is missing from "
            "diagnostics/london_may_sep_bedroom_varcomp.csv."
        )

    null_var = float(_vc_tbl.loc[null_key, "total_obs_var"])
    mod_var = float(v["total_obs_var"])
    pvar = 100.0 * (1.0 - mod_var / null_var)

    rows.append({
        "Model": m,
        "Null model": null_key,
        "Family": FAMILY[m],
        "ToD": TODS[m],
        "sigma2_u0": float(v["sig_u0"])**2,
        "sigma2_u1": float(v["sig_u1"])**2,
        "cov_u0u1": float(v["cov_u01"]),
        "sigma2_eps": float(v["sig_eps"])**2,
        "Total obs sigma2": mod_var,
        "% accounted for": pvar,
    })

tbl = pd.DataFrame(rows).set_index("Model")

display(
    tbl.style
    .format({
        "sigma2_u0": "{:.4f}",
        "sigma2_u1": "{:.4f}",
        "cov_u0u1": "{:+.4f}",
        "sigma2_eps": "{:.4f}",
        "Total obs sigma2": "{:.4f}",
        "% accounted for": "{:+.1f}%",
    })
    .set_caption(
        "Variance components and percentage variance accounted for relative to "
        "the family-specific constant-only null model. Total obs sigma2 is the "
        "observation-level variance exported from R: it includes the random "
        "intercept, random slope, their covariance, the observed distribution of "
        "T_out_c, and the residual variance scale from nlme. sigma2_eps is shown "
        "for reporting, but total_obs_var is the quantity used for the percentage calculation."
    )
)
